In [6]:
!apt-get install -y cuda-cudart-13-0
!wget https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb
!dpkg -i cuda-keyring_1.1-1_all.deb
!apt-get update && apt-get install -y cuda-cudart-13-0

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
cuda-cudart-13-0 is already the newest version (13.0.96-1).
0 upgraded, 0 newly installed, 0 to remove and 71 not upgraded.
--2026-05-25 19:19:07--  https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb
Resolving developer.download.nvidia.com (developer.download.nvidia.com)... 23.40.40.83, 23.40.40.73
Connecting to developer.download.nvidia.com (developer.download.nvidia.com)|23.40.40.83|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4332 (4.2K) [application/x-deb]
Saving to: ‘cuda-keyring_1.1-1_all.deb.2’

cuda-keyring_1.1-1_ 100%[===================>]   4.23K  --.-KB/s    in 0s      

2026-05-25 19:19:09 (1.81 GB/s) - ‘cuda-keyring_1.1-1_all.deb.2’ saved [4332/4332]

(Reading database ... 122380 files and directories currently installed.)
Preparing to unpack cuda-keyring_1.1-1_all.deb ...
Unpacking cuda-keyring (1.

In [7]:
!nvidia-smi

Mon May 25 19:19:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   34C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
!pip install --upgrade pip
!pip install wandb hugginface_hub
!pip install vllm --extra-index-url https://download.pytorch.org/whl/cu129

ERROR: Could not find a version that satisfies the requirement hugginface_hub (from versions: none)
ERROR: No matching distribution found for hugginface_hub
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu129


In [9]:
from google.colab import drive
drive.mount('/content/drive')

import os
RESULTS_DIR = '/content/drive/MyDrive/resilient_results'
os.makedirs(RESULTS_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
from huggingface_hub import login
import os
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
wandb_api = userdata.get("WANDB_API")

login(hf_token)  # paste your HF token

import wandb
wandb.login(key=wandb_api)  # paste your W&B token

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: saesha-parekh (saesha-parekhcivicdatalab) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [25]:
import os
import time
import requests # For checking server status
import subprocess

# 1. Kill any existing vLLM processes to ensure a clean start
print("Killing any existing vLLM server processes...")
# Use 'pkill -f' to find and terminate processes by command line
subprocess.run(["pkill", "-f", "vllm serve"], check=False)
time.sleep(2) # Give a moment for processes to terminate

# 2. Disable FlashInfer JIT sampling (requires nvcc which is not available on this runtime)
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

# Start vLLM server in the background using nohup and &
# Redirect stdout and stderr to a log file for debugging
vllm_log_file = "vllm_server.log"
vllm_command = f"""
nohup vllm serve google/gemma-4-E4B-it \
  --max-model-len 8192 \
  --gpu-memory-utilization 0.90 \
  --dtype float16 \
  --trust-remote-code \
  --port 8000 \
  > {vllm_log_file} 2>&1 &
"""
print(f"Starting vLLM server (output in {vllm_log_file})...")
# Execute the command directly in the shell
!{vllm_command}

# 3. Implement a robust waiting mechanism
# Check the log file for startup message or attempt a connection
server_ready = False
max_wait_time = 300 # Maximum wait time in seconds (5 minutes)
start_time = time.time()
print("Waiting for vLLM server to become ready (checking log and port)...")

while not server_ready and (time.time() - start_time) < max_wait_time:
    # Check log file for "Application startup complete"
    if os.path.exists(vllm_log_file):
        with open(vllm_log_file, "r") as f:
            log_content = f.read()
            if "Application startup complete" in log_content:
                print("Found 'Application startup complete' in log file.")
                # The log might show startup complete before the API is fully responsive,
                # so we continue to check the API endpoint.
                server_ready = True # Tentatively mark as ready from logs

    # Also try to hit a known API endpoint to confirm readiness
    try:
        # Check /v1/models endpoint, which typically lists available models
        response = requests.get("http://localhost:8000/v1/models", timeout=5)
        if response.status_code == 200:
            print("Successfully connected to vLLM server's /v1/models endpoint.")
            server_ready = True
            break # Exit loop if connection successful
    except requests.exceptions.ConnectionError:
        pass # Server not yet ready to accept connections
    except requests.exceptions.Timeout:
        pass # Connection timed out, still not ready
    except Exception as e:
        # Catch other potential request errors
        print(f"Warning: Error checking vLLM server status: {e}")

    if not server_ready:
        time.sleep(5) # Wait 5 seconds before checking again

if server_ready:
    print("✅ vLLM server is now ready.")
    # Optionally, print the last few lines of the log for confirmation
    if os.path.exists(vllm_log_file):
        with open(vllm_log_file, "r") as f:
            lines = f.readlines()
            print("--- Last 10 lines of vLLM Server Log ---")
            for line in lines[-10:]:
                print(line, end="")
            print("---------------------------------------")
else:
    print("❌ vLLM server did not become ready within the allotted time.")
    if os.path.exists(vllm_log_file):
        with open(vllm_log_file, "r") as f:
            print("--- Full vLLM Server Log ---")
            print(f.read())
            print("----------------------------")
    # If the server didn't start, we might want to kill it again just in case
    subprocess.run(["pkill", "-f", "vllm serve"], check=False)


Killing any existing vLLM server processes...
Starting vLLM server (output in vllm_server.log)...
Waiting for vLLM server to become ready (checking log and port)...
Found 'Application startup complete' in log file.
Successfully connected to vLLM server's /v1/models endpoint.
✅ vLLM server is now ready.
--- Last 10 lines of vLLM Server Log ---
(APIServer pid=22671) INFO 05-25 19:44:03 [launcher.py:46] Route: /inference/v1/generate, Methods: POST
(APIServer pid=22671) INFO 05-25 19:44:03 [launcher.py:46] Route: /scale_elastic_ep, Methods: POST
(APIServer pid=22671) INFO 05-25 19:44:03 [launcher.py:46] Route: /is_scaling_elastic_ep, Methods: POST
(APIServer pid=22671) INFO 05-25 19:44:03 [launcher.py:46] Route: /generative_scoring, Methods: POST
(APIServer pid=22671) INFO 05-25 19:44:03 [launcher.py:46] Route: /v1/chat/completions/render, Methods: POST
(APIServer pid=22671) INFO 05-25 19:44:03 [launcher.py:46] Route: /v1/completions/render, Methods: POST
(APIServer pid=22671) INFO:     St

In [16]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY"   # vLLM doesn't enforce a key
)

response = client.chat.completions.create(
    model="google/gemma-4-E4B-it",
    messages=[{"role": "user", "content": "Explain PagedAttention briefly."}],
    max_tokens=512,
    temperature=1
)

print(response.choices[0].message.content)

## PagedAttention Explained Briefly

**PagedAttention** is a memory management technique designed specifically for **Large Language Models (LLMs)** to make them significantly more memory-efficient and faster when processing long sequences (i.e., handling long context windows).

Here's the breakdown:

**The Problem (Traditional Attention/KV Cache):**

* When an LLM processes text, it needs to store the **Key ($\text{K}$)** and **Value ($\text{V}$)** vectors for every token it has already seen. This storage is called the **KV Cache**.
* Traditionally, the KV Cache is allocated in **contiguous blocks** of memory. If a sequence has a length of 100, it must reserve a single, unbroken chunk of memory large enough for 100 tokens, even if the tokens arrive unevenly or are being processed in small batches. This leads to massive **memory fragmentation** and wasted space.

**The Solution (PagedAttention):**

* PagedAttention borrows the concept of **paging** from virtual memory management in oper

In [17]:
!git clone https://github.com/EvolvingLMMs-Lab/lmms-eval.git
!cd lmms-eval && uv pip install -e ".[all]"

Cloning into 'lmms-eval'...
remote: Enumerating objects: 22318, done.
remote: Counting objects: 100% (565/565), done.
remote: Compressing objects: 100% (236/236), done.
remote: Total 22318 (delta 400), reused 345 (delta 329), pack-reused 21753 (from 2)
Receiving objects: 100% (22318/22318), 14.45 MiB | 17.54 MiB/s, done.
Resolving deltas: 100% (13853/13853), done.
Using Python 3.12.13 environment at: /usr
Resolved 267 packages in 4.29s
Prepared 70 packages in 2.54s
Uninstalled 5 packages in 59ms
Installed 70 packages in 52ms
 + anls==0.0.2
 - antlr4-python3-runtime==4.9.3
 + antlr4-python3-runtime==4.7.2
 + av==15.1.0
 + black==26.5.1
 + capture-metric==0.1.13
 + cbor==1.0.0
 + cfgv==3.5.0
 + colorama==0.4.6
 + dataproperty==1.1.1
 + distlib==0.4.0
 + duckduckgo-search==8.1.1
 + evaluate==0.4.6
 + factualscenegraph==0.6.1
 + flagembedding==1.4.0
 + ftfy==6.3.1
 + hf-transfer==0.1.9
 - httpx-sse==0.4.3
 + httpx-sse==0.4.0
 + identify==2.6.19
 + inscriptis==2.6.0
 + ir-datasets==0.5.11
 

In [18]:
!pip install decord
!python -m lmms_eval --tasks list

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 155.6 MB/s  0:00:00
2026-05-25 19:27:17 | INFO     | lmms_eval.__main__:cli_evaluate:475 - Verbosity set to INFO
2026-05-25 19:27:21 | INFO     | lmms_eval.__main__:cli_evaluate_single:594 - Evaluation tracker args: {}
2026-05-25 19:27:21 | INFO     | lmms_eval.__main__:cli_evaluate_single:640 - Available Tasks:
 - 3dsrbench
 - 3dsrbench_circular
 - ConBench
 - FALCONBench_mcq
 - FALCONBench_mcq_temploc
 - FALCONBench_oq
 - FALCONBench_oq_temploc
 - JumpScore
 - VisualPuzzles_cot
 - VisualPuzzles_direct
 - WISE
 - abench_dev
 - activitynetqa
 - ai2_arc
 - ai2d
 - ai2d_lite
 - ai2d_no_mask
 - ai2d_reasoning
 - aime24_agg8_reasoning
 - aime24_figures
 - aime24_figures_agg64
 - aime24_nofigures
 - aime24_nofigures_agg64
 - aime24_nofigures_agg8
 - aime25_agg8_reasoning
 - aime25_nofigures
 - aime25_nofigures_agg64
 - aime25_nofigures_agg8
 - aime_2024_agg8
 - aime_2024_rebase
 - aime_figures
 - aime_nofigures
 - aime_reasoning
 - ai

In [26]:
import time
time.sleep(30) # Give vLLM server more time to stabilize
!python -m lmms_eval \
  --model openai \
  --model_args model=google/gemma-4-E4B-it,base_url=http://localhost:8000/v1,api_key=dummy,request_timeout=600 \
  --tasks mmmu_pro \
  --batch_size 10 \
  --wandb_args project='saesha-parekhcivicdatalab-Gemma-4-Compression',name=mmmu_pro_run \
  --verbosity=DEBUG \
  --output_path ./results

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: saesha-parekh (saesha-parekhcivicdatalab) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ setting up run 957o1gb9 (0.1s)
wandb: ⣽ setting up run 957o1gb9 (0.1s)
wandb: ⣾ setting up run 957o1gb9 (0.1s)
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /content/wandb/run-20260525_194518-957o1gb9
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run mmmu_pro_run
wandb: ⭐️ View project at https://wandb.ai/saesha-parekhcivicdatalab/saesha-parekhcivicdatalab-Gemma-4-Compression
wandb: 🚀 View run at https://wandb.ai/saesha-parekhcivicdatalab/saesha-parekhcivicdatalab-Gemma-4-Compression/runs/957o1gb9
2026-05-25 19:45:20 | INFO     | lmms_eval.__main__:cli_evaluate:475 - Verbosity set to DEBUG
2026-05-25 19:45:20 | DEBUG    | lmms_eval.tasks:_get_task_and_group:458 - 